In [1]:
import os
import sys
import matplotlib

import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

import tacco as tc

# Variables

In [2]:
data_object_dir = '/rfs/project/rfs-iCNyzSAaucw/kk837/data_objects/Foetal/VisiumHD/Revision_Oct2025'
!ls {data_object_dir}

all_b2c_cells_filtered_celltype-sel_raw.h5ad
all_b2c_cells_filtered_raw.h5ad
all_b2c_cells_raw.h5ad
archive_17Nov2025
epicardium-mixture_b2c_cells_filtered_raw.h5ad
erythrocytes_b2c_cells_filtered_raw.h5ad
scVI
TACCO
unclassified-mixture_b2c_cells_filtered_raw.h5ad


# Read in reference data

In [3]:
ref = sc.read_h5ad(f'{data_object_dir}/TACCO/reference_raw.h5ad')
ref

AnnData object with n_obs × n_vars = 302702 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2', 'reference_type', 'latent_RT_efficiency', 'latent_cell_probability', 'latent_scale', 'sangerID', 'combinedID', 'region', 'age', 'facility', 'cell_or_nuclei', 'modality', 'kit_10x', 'scrublet_score', 'doublet_pval', 'doublet_bh_pval', 'n_counts', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'HB_score', 'donor_by_library-prep', 'multiplexed', 'SOC | status', 'SOC | log_prob_si

# Read in query data

In [4]:
que = sc.read_h5ad(f'{data_object_dir}/all_b2c_cells_filtered_celltype-sel_raw.h5ad')
que

AnnData object with n_obs × n_vars = 217420 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_2_colors', 'coarse_grain_pre_colors', 'donor_colors', 'library_colors', 'neighbors', 'spatial', 'umap'
    obsm: 'spatial', 'spatial_cropped_150_buffer'
    obsp: 'connectivities', 'dist

# Step 1: spot deconvolution

In [5]:
%%time
tc.tl.annotate(
    que,
    ref, 
    'reference_celltype',
    result_key='TACCO_ref-HD-SCN',
    multi_center=10, # how many subpopulation would you expect in a cell type label
    reconstruction_key='rec_ref-HD-SCN',
    counts_location=('X'),
    assume_valid_counts=True
)

Starting preprocessing
Annotation profiles were not found in `reference.varm["reference_celltype"]`. Constructing reference profiles with `tacco.preprocessing.construct_reference_profiles` and default arguments...
Finished preprocessing in 69.68 seconds.
Starting annotation of data with shape (217420, 18077) and a reference of shape (302702, 18077) using the following wrapped method:
+- platform normalization: platform_iterations=0, gene_keys=reference_celltype, normalize_to=adata
   +- multi center: multi_center=10 multi_center_amplitudes=True
      +- bisection boost: bisections=4, bisection_divisor=3
         +- core: method=OT annotation_prior=None
mean,std( rescaling(gene) )  0.5465582891820369 0.3527392624815032
bisection run on 1
bisection run on 0.6666666666666667
bisection run on 0.4444444444444444
bisection run on 0.2962962962962963
bisection run on 0.19753086419753085
bisection run on 0.09876543209876543
Finished annotation in 232.14 seconds.
CPU times: user 12min 10s, sys: 

AnnData object with n_obs × n_vars = 217420 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_2_colors', 'coarse_grain_pre_colors', 'donor_colors', 'library_colors', 'neighbors', 'spatial', 'umap', 'rec_ref-HD-SCN'
    obsm: 'spatial', 'spatial_cropped_150_buffer', 'rec_ref-HD-SC

In [6]:
que.obsm['TACCO_ref-HD-SCN']

reference_celltype,AtrialCardiomyocytes,EndothelialCells,EpicardialCells,LymphaticEndothelialCells,LymphoidCells,MesenchymalCells,MyeloidCells,NeuralCells,VentricularCardiomyocytes
HEA_FOET14880396___1,5.955057e-04,7.351747e-04,1.404406e-03,0.007122,0.870447,0.108628,0.009803,9.758398e-04,2.889895e-04
HEA_FOET14880396___2,1.524349e-03,8.613416e-03,1.678343e-03,0.112087,0.804112,0.065021,0.003731,2.355699e-03,8.770495e-04
HEA_FOET14880396___3,3.348211e-07,4.655622e-08,1.421907e-07,0.000006,0.063490,0.000001,0.936502,8.803284e-08,2.138997e-07
HEA_FOET14880396___5,3.988370e-06,4.264864e-03,3.670749e-03,0.010286,0.000041,0.978713,0.000389,2.628022e-03,2.605237e-06
HEA_FOET14880396___6,1.939675e-04,3.376934e-03,6.387251e-02,0.017243,0.101182,0.801681,0.003090,8.976677e-03,3.845630e-04
...,...,...,...,...,...,...,...,...,...
CellTalkHHD_Human_13_BRC2757-2758-7___289086,7.020798e-01,1.280856e-06,6.996058e-06,0.000004,0.000690,0.000055,0.000042,3.716023e-05,2.970842e-01
CellTalkHHD_Human_13_BRC2757-2758-7___289093,6.464092e-02,1.534633e-03,1.192455e-01,0.000796,0.121836,0.645423,0.008169,9.498083e-03,2.885735e-02
CellTalkHHD_Human_13_BRC2757-2758-7___289189,6.747167e-01,1.281959e-05,1.015906e-04,0.000010,0.001322,0.000554,0.000066,4.682171e-05,3.231704e-01
CellTalkHHD_Human_13_BRC2757-2758-7___289214,8.237117e-01,1.510793e-04,6.940851e-05,0.000469,0.058858,0.000369,0.009915,1.983760e-04,1.062585e-01


# Step 2: split the object

In [7]:
%%time
# splitting
sdata = tc.tl.split_observations(
    que, 
    'rec_ref-HD-SCN', 
    map_obs_keys=True, 
    result_key='coarse_grain_tacco_ref-hd-scn',
    counts_location=('X'),
)

removed 8 of 18085 genes from count matrix due to zero counts in gene
removed 8 of 18085 genes from profile definition due to zero appearance in the profiles
scale.....time 247.87106108665466
fuseall...time 117.67077684402466
CPU times: user 8min 42s, sys: 2min 21s, total: 11min 4s
Wall time: 6min 22s


In [8]:
que.obs

,object_id,bin_count,array_row,array_col,labels_joint_source,in_tissue_manual,library,donor_section_ID,per-frame_donorIDs,donor,...,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,library_donor,n_genes,coarse_grain_pre,coarse_grain_pre_2
HEA_FOET14880396___1,1,31,2932.612903,334.000000,primary,tissue,HEA_FOET14880396,NaN,C194,C194,...,21.179775,32.415730,49.887640,32.0,3.496508,1.797753,HEA_FOET14880396__C194,1392,Mesenchymal,unclassified_mixture_Mesenchymal-LEC-Immune
HEA_FOET14880396___2,2,26,2959.346154,336.923077,primary,tissue,HEA_FOET14880396,NaN,C194,C194,...,19.337267,28.415797,46.618248,89.0,4.499810,4.039946,HEA_FOET14880396__C194,1676,Mesenchymal,unclassified_mixture_Mesenchymal-LEC-Immune
HEA_FOET14880396___3,3,32,2882.562500,334.187500,primary,tissue,HEA_FOET14880396,NaN,C194,C194,...,25.690236,34.848485,52.659933,84.0,4.442651,2.828283,HEA_FOET14880396__C194,1906,unclassified_mixture,MyeloidCells
HEA_FOET14880396___5,5,35,2870.714286,305.171429,primary,tissue,HEA_FOET14880396,NaN,C194,C194,...,35.676626,46.309315,72.671353,23.0,3.178054,2.021090,HEA_FOET14880396__C194,811,Mesenchymal,MesenchymalCells
HEA_FOET14880396___6,6,31,2877.612903,308.000000,primary,tissue,HEA_FOET14880396,NaN,C194,C194,...,38.461538,56.350626,100.000000,30.0,3.433987,5.366726,HEA_FOET14880396__C194,444,unclassified_mixture,unclassified_mixture
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CellTalkHHD_Human_13_BRC2757-2758-7___289086,289086,5,1019.200000,2912.600000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,...,68.393782,100.000000,100.000000,6.0,1.945910,3.108808,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,161,AtrialCardiomyocytes,AtrialCardiomyocytes
CellTalkHHD_Human_13_BRC2757-2758-7___289093,289093,7,1087.285714,2794.428571,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,...,50.909091,96.363636,100.000000,2.0,1.098612,0.909091,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,208,VentricularCardiomyocytes,MesenchymalCells
CellTalkHHD_Human_13_BRC2757-2758-7___289189,289189,5,1076.800000,2851.000000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,...,64.705882,100.000000,100.000000,9.0,2.302585,4.411765,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,172,AtrialCardiomyocytes,AtrialCardiomyocytes
CellTalkHHD_Human_13_BRC2757-2758-7___289214,289214,6,1093.000000,3095.833333,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",BRC2758,...,34.991424,52.144082,100.000000,9.0,2.302585,1.543739,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,479,AtrialCardiomyocytes,AtrialCardiomyocytes


In [9]:
sdata.obs

,index,object_id,bin_count,array_row,array_col,labels_joint_source,in_tissue_manual,library,donor_section_ID,per-frame_donorIDs,...,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,library_donor,n_genes,coarse_grain_pre,coarse_grain_pre_2,coarse_grain_tacco_ref-hd-scn
0,HEA_FOET14880396___1,1,31,2932.612903,334.000000,primary,tissue,HEA_FOET14880396,NaN,C194,...,32.415730,49.887640,32.0,3.496508,1.797753,HEA_FOET14880396__C194,1392,Mesenchymal,unclassified_mixture_Mesenchymal-LEC-Immune,AtrialCardiomyocytes
1,HEA_FOET14880396___2,2,26,2959.346154,336.923077,primary,tissue,HEA_FOET14880396,NaN,C194,...,28.415797,46.618248,89.0,4.499810,4.039946,HEA_FOET14880396__C194,1676,Mesenchymal,unclassified_mixture_Mesenchymal-LEC-Immune,AtrialCardiomyocytes
2,HEA_FOET14880396___6,6,31,2877.612903,308.000000,primary,tissue,HEA_FOET14880396,NaN,C194,...,56.350626,100.000000,30.0,3.433987,5.366726,HEA_FOET14880396__C194,444,unclassified_mixture,unclassified_mixture,AtrialCardiomyocytes
3,HEA_FOET14880396___14,14,29,2879.103448,362.517241,primary,tissue,HEA_FOET14880396,NaN,C194,...,37.500000,61.693548,25.0,3.258096,2.016129,HEA_FOET14880396__C194,975,Mesenchymal,MesenchymalCells,AtrialCardiomyocytes
4,HEA_FOET14880396___18,18,12,2921.166667,333.166667,primary,tissue,HEA_FOET14880396,NaN,C194,...,26.389444,43.862455,41.0,3.737670,1.639344,HEA_FOET14880396__C194,1904,unclassified_mixture,NeuralCells,AtrialCardiomyocytes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1123478,CellTalkHHD_Human_13_BRC2757-2758-7___289086,289086,5,1019.200000,2912.600000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",...,100.000000,100.000000,6.0,1.945910,3.108808,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,161,AtrialCardiomyocytes,AtrialCardiomyocytes,VentricularCardiomyocytes
1123479,CellTalkHHD_Human_13_BRC2757-2758-7___289093,289093,7,1087.285714,2794.428571,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",...,96.363636,100.000000,2.0,1.098612,0.909091,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,208,VentricularCardiomyocytes,MesenchymalCells,VentricularCardiomyocytes
1123480,CellTalkHHD_Human_13_BRC2757-2758-7___289189,289189,5,1076.800000,2851.000000,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",...,100.000000,100.000000,9.0,2.302585,4.411765,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,172,AtrialCardiomyocytes,AtrialCardiomyocytes,VentricularCardiomyocytes
1123481,CellTalkHHD_Human_13_BRC2757-2758-7___289214,289214,6,1093.000000,3095.833333,secondary,tissue,CellTalkHHD_Human_13_BRC2757-2758-7,BRC2758__2,"BRC2757,BRC2758",...,52.144082,100.000000,9.0,2.302585,1.543739,CellTalkHHD_Human_13_BRC2757-2758-7__BRC2758,479,AtrialCardiomyocytes,AtrialCardiomyocytes,VentricularCardiomyocytes


In [11]:
original_cell_id = 'CellTalkHHD_Human_13_BRC2757-2758-7___2'
sdata.obs[sdata.obs['index']==original_cell_id][['object_id','bin_count','array_row','array_col',
                                                 'coarse_grain_pre_2','coarse_grain_tacco_ref-hd-scn']]

,object_id,bin_count,array_row,array_col,coarse_grain_pre_2,coarse_grain_tacco_ref-hd-scn
117854,2,37,2456.243243,177.810811,AtrialCardiomyocytes,AtrialCardiomyocytes
251359,2,37,2456.243243,177.810811,AtrialCardiomyocytes,EndothelialCells
355714,2,37,2456.243243,177.810811,AtrialCardiomyocytes,EpicardialCells
455396,2,37,2456.243243,177.810811,AtrialCardiomyocytes,LymphaticEndothelialCells
551328,2,37,2456.243243,177.810811,AtrialCardiomyocytes,LymphoidCells
705820,2,37,2456.243243,177.810811,AtrialCardiomyocytes,MesenchymalCells
883449,2,37,2456.243243,177.810811,AtrialCardiomyocytes,NeuralCells
1047088,2,37,2456.243243,177.810811,AtrialCardiomyocytes,VentricularCardiomyocytes


--> TACCO creates multiple cells on a single coordinate. Doesn't deconvolute the bin

# Save data

In [12]:
print(sdata.X.data[:10])

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [14]:
# calculate QC metrices
sc.pp.calculate_qc_metrics(sdata, qc_vars=["mt"], inplace=True)

In [17]:
del que.uns['rec_ref-HD-SCN'] ## Need to remove - doesn't seem to contain anything important and causes issues when saving
que.write(f'{data_object_dir}/TACCO/all_b2c_cells_filtered_celltype-sel_post-tacco_raw.h5ad')
print(que.shape)
sdata.write(f'{data_object_dir}/TACCO/tacco-splitted_ref-HD-SCN_raw.h5ad')
print(sdata.shape)

(217420, 18085)
(1123483, 18077)


In [18]:
!ls -lh {data_object_dir}/TACCO

total 7.9G
-rwxrwx---+ 1 kk837 kk837 5.0G Dec  9 23:07 all_b2c_cells_filtered_celltype-sel_post-tacco_raw.h5ad
-rwxrwx---+ 1 kk837 kk837 3.3G Dec  9 21:53 reference_raw.h5ad
-rwxrwx---+ 1 kk837 kk837 2.9G Dec  9 22:46 tacco-splitted_ref-HD_raw.h5ad
-rwxrwx---+ 1 kk837 kk837 3.0G Dec  9 23:07 tacco-splitted_ref-HD-SCN_raw.h5ad


In [15]:
que

AnnData object with n_obs × n_vars = 217420 × 18085
    obs: 'object_id', 'bin_count', 'array_row', 'array_col', 'labels_joint_source', 'in_tissue_manual', 'library', 'donor_section_ID', 'per-frame_donorIDs', 'donor', 'CS', 'est_CS', 'GA', 'PCW', 'section_ID', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'library_donor', 'n_genes', 'coarse_grain_pre', 'coarse_grain_pre_2'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'PCW_colors', 'coarse_grain_pre_2_colors', 'coarse_grain_pre_colors', 'donor_colors', 'library_colors', 'neighbors', 'spatial', 'umap', 'rec_ref-HD-SCN'
    obsm: 'spatial', 'spatial_cropped_150_buffer', 'rec_ref-HD-SC